In [1]:
# 0.99 m x 0.5 m x 1 m tank dimensions 
# 48 electrodes (24 each on 2 opposite sides of tank) + 588 measurement protocols

In [2]:
import numpy as np

import pygimli as pg
import pygimli.meshtools as mt
from pygimli.physics import ert

In [3]:
# Neumann boundary conditions because current cant flow outisde boundaries (derivative at boundary = 0)

In [5]:
plc = mt.createCube(size=[0.99, 0.5, 1.0], pos=[0.495, 0.25], boundaryMarker=1) # plc is piecewise linear complex cube and pos is centre of said cube 

In [7]:
shm = pg.getExampleData("ert/modeltank.shm")

for s in shm.sensors():
    plc.createNode(s, marker=-99) #-99 marker is a special marker reserved for A, B, M, N

27/08/26 - 19:59:35 - pyGIMLi - INFO - Looking for ert/modeltank.shm in gimli-org/example-data/


In [8]:
plc.createNode([0.5, 0.5, -0.5], marker=-999) #-999 is for ensuring boundary condns and no current going out 

	ID: 56, Marker: -999	RVector3: (0.5, 0.5, -0.5)

In [9]:
plc.createNode([0.75, 0.25, 0.5], marker=-1000) #-1000 is the calibration node (arbitary value added at boundaries because neumann problems have only 
# derivative values at boundary (taking care of c in integeration))

	ID: 57, Marker: -1000	RVector3: (0.75, 0.25, 0.5)

In [11]:
for s in plc.positions(pg.find(plc.nodeMarkers() == -99)):
    plc.createNode(s - [0.0, 0.0, 1e-3])

# Also refine the reference node
plc.createNode([0.5, 0.5, -0.5 - 1e-3])
#pg.show(plc, markers=True, showMesh=True)

# nodes are inserted 1 mm away from each electrode in z dir. 

	ID: 106, Marker: 0	RVector3: (0.5, 0.5, -0.501)

In [12]:
mesh = mt.createMesh(plc)
pg.show(mesh, markers=True, showMesh=True)

28/08/26 - 18:13:25 - pyGIMLi - WARNING - Given data fits neither cell count nor node count:
28/08/26 - 18:13:25 - pyGIMLi - WARNING - 165272 vs. 0 vs. 5802


Widget(value='<iframe src="http://localhost:62366/index.html?ui=P_0x28541495550_0&reconnect=auto" class="pyvis…

(<pyvista.plotting.plotter.Plotter at 0x28541495550>, None)

In [13]:
# first make block for homogenous resisitivity of 1 ohm 
hom = ert.simulate(mesh, res=1.0, scheme=shm, sr=False,
                   calcOnly=True, verbose=True)

hom.save('homogeneous.ohm', 'a b m n u')

1

In [14]:
# now non homogeneity being added 
cube = mt.createCube(size=[0.3, 0.2, 0.8], pos=[0.7, 0.2], marker=2)
plc += cube

pg.show(plc, alpha=0.3)
mesh = mt.createMesh(plc)

Widget(value='<iframe src="http://localhost:62366/index.html?ui=P_0x28548b26350_1&reconnect=auto" class="pyvis…

In [15]:
# plc.exportVTK('plc')
# mesh.exportVTK('mesh')
pg.show(mesh, mesh.cellMarkers(), showMesh=True,
        filter={'clip':{'origin':(0.7, 0, 0.0)},})

Widget(value='<iframe src="http://localhost:62366/index.html?ui=P_0x28548cb0910_2&reconnect=auto" class="pyvis…

(<pyvista.plotting.plotter.Plotter at 0x28548cb0910>, None)

In [16]:
res = [[1, 10.0], [2, 100.0]]  # map markers 1 and 2 to 10 and 100 Ohmm, resp.
het = ert.simulate(mesh, res=res, scheme=shm, sr=False,
                   calcOnly=True, verbose=True)

In [17]:
het['k'] = 1.0/ (hom['u'] / hom['i'])
het['rhoa'] = het['k'] * het['u'] / het['i']

het.save('simulated.dat', 'a b m n rhoa k u i')

np.testing.assert_approx_equal(het['rhoa'][0], 9.5, 1)

# np.testing.assert_approx_equal(het('k')[0], 0.820615269548)